# Regression and Loss Choices

We fit the course's 16 polynomial observations using the same degree-four basis
and compare squared, absolute, and maximum residual losses. A row of the design
matrix is one observation; a column is one fixed basis function. The coefficients
are the decision variables.

Open the course repository root in VS Code, select the **Julia 1.12** kernel
with the course environment, and choose **Run All**. JuMP, HiGHS, and Plots
are already included; LinearAlgebra is part of Julia. No external data files
are needed.

The final plot also compares the fits after increasing one response by 12.
This outlier variant uses a separate copy of the response vector.

This demonstration follows the modeling book's material on regression losses.


In [ ]:
using JuMP, HiGHS, LinearAlgebra, Plots
raw_x = collect(1.0:16.0)
y = [1, 3, 0, 1, 2, 4, 6, 7, 5, 5, 6, 7.2, 5.5, 4, 3.2, 5]
# Fixed change of units for the known input interval [1,16].
scaled(t) = (t .- 8.5) ./ 7.5
design(t, degree) = hcat([scaled(t).^j for j in 0:degree]...)
A = design(raw_x, 4)  # intercept first
fine_x = collect(range(1, 16; length=250))


## One Design Matrix, Three Losses

For residuals $r=Au-y$, compare $\sum_i r_i^2$, $\sum_i|r_i|$, and
$\max_i|r_i|$. The latter two have LP formulations. Linear constraints on the
coefficients can be added to any version; for example, known nonnegative basis
weights are simply $u_j\geq0$ (when that assumption makes sense for the basis).

In [ ]:
function fit_loss(A, target, loss)
    m, n = size(A)
    model = Model(HiGHS.Optimizer)
    set_silent(model)
    @variable(model, u[1:n])
    @expression(model, r[i=1:m], sum(A[i,j]*u[j] for j in 1:n) - target[i])
    if loss == :squared
        @objective(model, Min, sum(r[i]^2 for i in 1:m))
    elseif loss == :absolute
        @variable(model, t[1:m] >= 0)
        @constraint(model, [i=1:m], r[i] <= t[i])
        @constraint(model, [i=1:m], -r[i] <= t[i])
        @objective(model, Min, sum(t))
    elseif loss == :maximum
        @variable(model, t >= 0)
        @constraint(model, [i=1:m], r[i] <= t)
        @constraint(model, [i=1:m], -r[i] <= t)
        @objective(model, Min, t)
    else
        error("Unknown loss")
    end
    optimize!(model)
    @assert is_solved_and_feasible(model)
    return value.(u), objective_value(model)
end
losses = [:squared, :absolute, :maximum]
fits = [fit_loss(A, y, loss) for loss in losses]
@assert norm(A*fits[1][1] - A*(A \ y)) < 1e-5
println("Objectives (different units): ", [fit[2] for fit in fits])


In [ ]:
function loss_panel(target, title_text)
    p = scatter(raw_x, target; label="Observations", color=:black,
        xlabel="Input", ylabel="Response", title=title_text, legend=:topleft)
    for (loss, label) in zip(losses, ["Squared loss", "Absolute loss", "Maximum loss"])
        u, _ = fit_loss(A, target, loss)
        plot!(p, fine_x, design(fine_x, 4)*u; label=label, linewidth=2)
    end
    p
end
outlier_y = copy(y)
outlier_y[8] += 12
loss_plot = plot(loss_panel(y, "Original observations"),
    loss_panel(outlier_y, "One response increased by 12");
    layout=(1,2), size=(1000,380), margin=5Plots.mm)
display(loss_plot)
